<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A;
             padding-bottom:.15em; margin-bottom:.5em; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size:.62em; border-radius:6px; border:1px solid #E2E6EE; box-shadow:none; }
.reveal pre code { max-height:420px; }
.reveal ul li, .reveal ol li { margin-bottom:.3em; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A;
        padding:.6em 1em; border-radius:0 6px 6px 0; margin:.5em 0; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E;
        padding:.6em 1em; border-radius:0 6px 6px 0; margin:.5em 0; }
</style>

## Generación de Variables Aleatorias
### Modelado de Sistemas bajo Incertidumbre · Semana 2
---
Departamento de Ingeniería Industrial · Universidad de los Andes

## Agenda
1. ¿Por qué generar variables aleatorias?
2. Números pseudoaleatorios uniformes — Generador Congruencial Lineal
3. Teorema de la Transformada Inversa
4. Ejemplos: Exponencial, Triangular
5. Implementación en Python con NumPy
6. Caso: Auditoría Casino Andino

## ¿Por Qué Generar Variables Aleatorias?
La simulación necesita **muestras controlables y reproducibles** de distribuciones de probabilidad.

**Requisitos de un buen generador:**
- Uniforme en $[0,1)$ — sin sesgos detectables
- **Período largo** — no repetirse pronto
- **Reproducible** — misma semilla → mismos números
- **Rápido** — millones de muestras en microsegundos

> Python/NumPy usa el algoritmo **Mersenne Twister** (período $2^{19937}-1$).  
> Entender sus bases es esencial para simulación rigurosa.

## Generador Congruencial Lineal (GCL)
<div class="defn">

$$X_{n+1} = (a \cdot X_n + c) \bmod m, \qquad U_n = X_n / m$$

- $X_0$: semilla (seed)
- $a$: multiplicador
- $c$: incremento
- $m$: módulo

</div>

**Ejemplo clásico (RANDU):** $a=65539$, $c=0$, $m=2^{31}$  
*Problema histórico: los triples $(U_n, U_{n+1}, U_{n+2})$ caen en solo 15 planos paralelos.*

In [ ]:
# GCL básico implementado a mano
def gcl(seed, a=1664525, c=1013904223, m=2**32, n=10):
    X = seed
    nums = []
    for _ in range(n):
        X = (a * X + c) % m
        nums.append(X / m)
    return nums

muestras = gcl(seed=42, n=5000)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(muestras, bins=30, color='#1A7A9A', edgecolor='white', alpha=.85)
axes[0].set_title('Distribución de U ~ GCL'); axes[0].set_xlabel('valor')
axes[1].scatter(muestras[:-1], muestras[1:], s=.5, c='#0D2240', alpha=.3)
axes[1].set_title('Pares consecutivos (U_n, U_{n+1})')
axes[1].set_xlabel('U_n'); axes[1].set_ylabel('U_{n+1}')
plt.tight_layout(); plt.show()

## Teorema de la Transformada Inversa
<div class="defn">

Si $U \sim \text{Uniforme}(0,1)$ y $F$ es la CDF de una variable continua $X$, entonces:

$$X = F^{-1}(U) \sim F$$

</div>

**Intuición geométrica:** muestrea un valor de probabilidad uniforme en $[0,1]$ y "mapéalo" a la distribución deseada usando la inversa de la CDF.

## Transformada Inversa: Distribución Exponencial
$$F(x) = 1 - e^{-\lambda x} \implies F^{-1}(u) = -\frac{\ln(1-u)}{\lambda}$$

Como $1-U \sim U$, usamos directamente:

$$X = -\frac{\ln U}{\lambda}$$

**Ejemplo:** tiempo entre llegadas de clientes con $\lambda = 2$ clientes/minuto

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import expon
np.random.seed(5)
lam = 2.0

U = np.random.uniform(0, 1, 5000)
X_inv = -np.log(U) / lam          # Transformada inversa
X_np  = np.random.exponential(1/lam, 5000)  # NumPy directo

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(X_inv, bins=50, alpha=.6, color='#1A7A9A', label='Transformada inversa', density=True)
ax.hist(X_np,  bins=50, alpha=.5, color='#C8961E', label='np.random.exponential', density=True)
x_t = np.linspace(0, 3, 200)
ax.plot(x_t, expon.pdf(x_t, scale=1/lam), 'k--', lw=1.5, label='PDF teórica')
ax.set_xlabel('x'); ax.set_ylabel('Densidad'); ax.legend()
ax.set_title(f'Exponencial(λ={lam}) — comparación de métodos')
plt.tight_layout(); plt.show()

## Caso: Auditoría Casino Andino
**Situación:** El casino afirma que su generador de dados es uniforme.  
Un auditor sospecha que ciertos resultados aparecen con más frecuencia.

**Hipótesis:**  
- $H_0$: Las probabilidades son iguales ($p_i = 1/6$)  
- $H_1$: Las probabilidades no son iguales

**Prueba $\chi^2$ de bondad de ajuste:**
$$\chi^2 = \sum_{i=1}^k \frac{(O_i - E_i)^2}{E_i}$$

In [ ]:
import numpy as np
from scipy.stats import chisquare
np.random.seed(99)

# Simulamos el generador "sospechoso" del casino
n_lanzamientos = 3000
# La cara 6 aparece con el doble de probabilidad
probs_casino = [1/7, 1/7, 1/7, 1/7, 1/7, 2/7]
dados = np.random.choice([1,2,3,4,5,6], size=n_lanzamientos, p=probs_casino)

observado = np.array([(dados == i).sum() for i in range(1, 7)])
esperado  = np.full(6, n_lanzamientos / 6)

stat, p_valor = chisquare(f_obs=observado, f_exp=esperado)
print(f"Frecuencias observadas: {observado}")
print(f"Frecuencia esperada:    {int(esperado[0])} por cara")
print(f"Estadístico χ²: {stat:.2f}")
print(f"p-valor: {p_valor:.4f}")
print()
if p_valor < 0.05:
    print("Conclusión: Se rechaza H0. El generador NO es uniforme.")
else:
    print("Conclusión: No hay evidencia suficiente para rechazar H0.")

## Conclusiones
- Los generadores pseudoaleatorios producen **secuencias deterministas** que imitan la aleatoriedad
- La **Transformada Inversa** es el método universal: si conoces $F^{-1}$, puedes simular cualquier distribución
- En Python, `numpy.random` implementa algoritmos robustos con períodos muy largos
- Las pruebas estadísticas (como $\chi^2$) permiten **auditar** si un generador es realmente uniforme